In [15]:
import numpy as np
import pandas as pd
import scanpy as sc
import sys
from statsmodels import robust
import matplotlib.pyplot as plt
import os.path
import anndata as ad
import seaborn as sns
import matplotlib as mpl
import os
import harmonypy
# import scvi

sc.set_figure_params(scanpy=True, dpi=80, dpi_save=150)
pd.set_option('display.max_columns', None)


C:\Users\sambe\anaconda3\envs\research23\lib\site-packages\anndata\core\anndata.py:17: FutureWarning: pandas.core.index is deprecated and will be removed in a future version. The public classes are available in the top-level namespace.
  from pandas.core.index import RangeIndex


ImportError: cannot import name 'PathLike' from 'anndata.compat' (C:\Users\sambe\anaconda3\envs\research23\lib\site-packages\anndata\compat\__init__.py)

In [2]:
os.chdir("C:/Users/sambe/OneDrive - Newcastle University/Research projects/Github/masters_newcastle/")
figures = "C:/Users/sambe/OneDrive - Newcastle University/Research projects/Github/masters_newcastle/scripts"
bigdata = "C:/Users/sambe/Documents/bigdata"
datas = "C:/Users/sambe/OneDrive - Newcastle University/Research projects/Github/masters_newcastle/data"
writes = "C:/Users/sambe/OneDrive - Newcastle University/Research projects/Github/masters_newcastle/write"
os.getcwd()

'C:\\Users\\sambe\\OneDrive - Newcastle University\\Research projects\\Github\\masters_newcastle'

In [4]:
pan_immune_bcells_harm = sc.read('data//pan_immune_bcells_harm.h5ad')
display(pan_immune_bcells_harm)
#pan_immune_pcells = sc.read(bigdata + "pan_immune_pcells.h5ad")
#display(pan_immune_pcells)
#pan_stroma_cells  = sc.read(bigdata + "pan_stroma_cells.h5ad")
#display(pan_stroma_cells)

AnnData object with n_obs × n_vars = 63942 × 1469
    obs: 'n_counts', 'n_genes', 'file', 'mito', 'doublet_scores', 'predicted_doublets', 'old_annotation_uniform', 'organ', 'Sort_id', 'age', 'method', 'donor', 'sex', 'Sample', 'scvi_clusters', 'is_maternal_contaminant', 'anno_lvl_2_final_clean', 'celltype_annotation', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'leiden'
    var: 'GeneID', 'GeneName', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'scvi_model_var', 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'mean', 'std'
    uns: 'celltype_annotation_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'method_colors', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

In [5]:
pan_immune_bcells_harm.X.log2().sum(axis = 1)

AttributeError: 'numpy.ndarray' object has no attribute 'log2'

In [6]:
display(pan_immune_bcells_harm.obs.organ.unique())

''' YS, yolk sac; LI, liver; BM, bone marrow; TH, thymus; SP, spleen; MLN, mesenteric lymph node; SK, skin; GU, gut; KI, kidney. '''

['SK', 'SP', 'YS', 'LI', 'TH', 'GU', 'BM', 'KI', 'MLN']
Categories (9, object): ['BM', 'GU', 'KI', 'LI', ..., 'SK', 'SP', 'TH', 'YS']

' YS, yolk sac; LI, liver; BM, bone marrow; TH, thymus; SP, spleen; MLN, mesenteric lymph node; SK, skin; GU, gut; KI, kidney. '

## Marker genes

In [ ]:
# the marker genes for my replication of the research [Mapping the developing human immune system across organs]
# These have been gathered from the work of jardine et al [Blood and immune development in human fetal bone marrow and Down syndrome] present in supplementary table 6 of that study.
# and the work of Popescu et al [Decoding human fetal liver haematopoiesis] in supplementary table 3
# the combination of these two made up the marker genes for my main replication paper.
# main article used SCVI so i should probably use that too.
# monocle is a trajectory package in python that is the standard for producing trajectories

# found the github for main paper (https://github.com/Teichlab/Pan_fetal_immune/tree/master/metadata/marker_genes) 
#contains files with the marker genes used.


In [8]:
markers = pd.read_csv('https://raw.githubusercontent.com/Teichlab/Pan_fetal_immune/ef0241a2a91e40dc643f31e2b0e1cebc46684258/metadata/marker_genes/B_marker_genes_09072021_clean.csv',index_col=0,parse_dates=[0])
# error until I read the file in from raw.

In [9]:

display(markers.loc[markers['anno_lvl_2'] == 'HSC_MPP'])

display(markers.loc[markers['gene'] == 'CD34'])


,gene,anno_lvl_2
0,CLEC9A,HSC_MPP
1,CD34,HSC_MPP
4,SPINK2,HSC_MPP
10,KIT,HSC_MPP
13,FLT3,HSC_MPP


,gene,anno_lvl_2
1,CD34,HSC_MPP
2,CD34,CMP
3,CD34,ELP


In [10]:
markername = markers['anno_lvl_2'].unique()
genename = markers['gene'].unique()

#display(markers.loc[markers['gene'] == 'CD34'])

# next goal is to mutate a column in main data that aligns these cell type annotations ot the relevant genes and does
# so without bias (i remember a discussion somewhere about automatically applying labels to the data based on highest % of expressed gene)


In [11]:
bcells = ['PRE_PRO_B', 'PRE_PRO_B_CELL', 'PRE_PRO_B',  'LATE_PRO_B','PRO_B','LARGE_PRE_B','SMALL_PRE_B', 'IMMATURE_B','MATURE_B','B1', 'CYCLING_B']
bcells

['PRE_PRO_B',
 'PRE_PRO_B_CELL',
 'PRE_PRO_B',
 'LATE_PRO_B',
 'PRO_B',
 'LARGE_PRE_B',
 'SMALL_PRE_B',
 'IMMATURE_B',
 'MATURE_B',
 'B1',
 'CYCLING_B']

In [58]:
markername

array(['HSC_MPP', 'CMP', 'ELP', 'PRE PRO B CELL', 'PRO B CELL',
       'LATE PRO B CELL', 'PRO TO PRE B CELL', 'LARGE PRE B CELL',
       'SMALL PRE B CELL', 'IMMATURE B CELL', 'MATURE B CELL',
       'CYCLING B/B1 CELL', 'PLASMA B CELL'], dtype=object)

In [64]:
markersoup = markers.set_index('anno_lvl_2')
markersoup.index.name = "an"

In [68]:
markersoup

,gene
an,
HSC_MPP,CLEC9A
HSC_MPP,CD34
CMP,CD34
ELP,CD34
HSC_MPP,SPINK2
...,...
CYCLING B/B1 CELL,CD5
CYCLING B/B1 CELL,SPN
PLASMA B CELL,JCHAIN


In [73]:
d = {}
for x in marker:
    y = markers.loc[markers['anno_lvl_2'] == x] 
    d[x] = y["gene"]
    

NameError: name 'marker' is not defined

In [74]:
os.getcwd()

'C:\\Users\\sambe\\OneDrive - Newcastle University\\Research projects\\Github\\masters_newcastle'

In [77]:
markersoup.to_csv(datas + "marker_genes.xls")

In [59]:
markers.loc[markers['anno_lvl_2'] == "CYCLING B/B1 CELL"]

,gene,anno_lvl_2
25,CD19,CYCLING B/B1 CELL
47,MKI67,CYCLING B/B1 CELL
49,CD27,CYCLING B/B1 CELL
62,MS4A1,CYCLING B/B1 CELL
71,CD24,CYCLING B/B1 CELL
85,CD40,CYCLING B/B1 CELL
89,FCER2,CYCLING B/B1 CELL
91,CD5,CYCLING B/B1 CELL
92,SPN,CYCLING B/B1 CELL


In [46]:
for x in markername:
   markers.loc[markers['anno_lvl_2'] == x]


In [ ]:
for x in genename:
    display(markers.loc[markers['gene'] == x])

In [ ]:
''' ARCHIVE

paper_cells_list = markers["anno_lvl_2"].unique().tolist()
paper_cells = []
for i in paper_cells_list:
    j = i.replace(' ','_')
    paper_cells.append(j)
    
ARCHIVE


for x in markername:
    a_data = markers.loc[markers['anno_lvl_2'] == x]
    for y in paper_cells:
        y = a_data.gene.tolist()
    
    #for y in a_data:
#        y.gene.tolist()

THIS DATA ARCHIVED UNTIL DICTIONARY TEST
'''

In [ ]:
'''for x in markername:
    a_data = markers.loc[markers['anno_lvl_2'] == x]

for x in markername:
    a_data = markers.loc[markers['anno_lvl_2'] == x]
    paper_cells = a_data.gene.to_list()
    '''

In [12]:
dictionarymarkers = {
    'CMP': ['CD34','SPINK2','MPO','CSF1R','KIT','FLT3'],
    'CYCLING B/B1 CELL': ['CD19','MKI67','CD27','MS4A1','CD24','CD40','FCER2','CD5','SPN'],
    'ELP': ['CD34','SPINK2','IL7R','KIT','FLT3',],
    'HSC_MPP': ['CLEC9A','CD34','SPINK2','KIT','FLT3'],
    'IMMATURE B CELL': ['CD19','MS4A1','CD24','SPIB','CD40','FCER2'],
    'LARGE PRE B CELL': ['CD19','VPREB1','MME','CDC45','DHFR','MKI67','RAG1','CD24','TNFRSF17','MME','IDH2'],
    'LATE PRO B CELL': ['CD19','VPREB1','MME','CD27','RAG1','DNTT','CD24','MME'],
    'MATURE B CELL': ['CD19','MS4A1','CD24','CD40','FCER2'],
    'PLASMA B CELL': ['CD19','CD27','MS4A1','TNFRSF17','CD40','FCER2','JCHAIN','SDC1','CD38'],
    'PRE PRO B CELL': ['FLT3','CD19','VPREB1','RAG1','MS4A1'],
    'PRO B CELL': ['CD19','VPREB1','MME','CDC45','DHFR','MKI67','RAG1','DNTT','CD24','MME'],
    'PRO TO PRE B CELL': ['CD19','VPREB1','MME','CDC45','DHFR','MKI67','MS4A1','CD24','MME','IDH2'],
    'SMALL PRE B CELL': ['CD19','VPREB1','MME','RAG1','CD24','TNFRSF17','MME']}

In [16]:
pd.DataFrame(dictionarymarkers).to_csv(bigdata + 'dictionarymarkers.csv', index=False)

ValueError: All arrays must be of the same length

In [9]:
thisdict = {
  "brand": "Ford",
  "model": "Mustang",
  "year": 1964,
  "year": [2020,777,333]
}
print(thisdict)

{'brand': 'Ford', 'model': 'Mustang', 'year': [2020, 777, 333]}


In [10]:
print(len(thisdict))
thisdict['year']

3


[2020, 777, 333]

In [13]:
dictionarymarkers['CYCLING B/B1 CELL']

['CD19', 'MKI67', 'CD27', 'MS4A1', 'CD24', 'CD40', 'FCER2', 'CD5', 'SPN']

In [24]:
markers.sort_values('anno_lvl_2')

,gene,anno_lvl_2
2,CD34,CMP
14,FLT3,CMP
5,SPINK2,CMP
7,MPO,CMP
8,CSF1R,CMP
...,...,...
32,VPREB1,SMALL PRE B CELL
55,RAG1,SMALL PRE B CELL
22,CD19,SMALL PRE B CELL
68,CD24,SMALL PRE B CELL


In [30]:
markers

,gene,anno_lvl_2
0,CLEC9A,HSC_MPP
1,CD34,HSC_MPP
2,CD34,CMP
3,CD34,ELP
4,SPINK2,HSC_MPP
...,...,...
91,CD5,CYCLING B/B1 CELL
92,SPN,CYCLING B/B1 CELL
93,JCHAIN,PLASMA B CELL
94,SDC1,PLASMA B CELL


In [60]:
markersoup = markers.set_index('anno_lvl_2')

In [61]:
markersoup

,gene
anno_lvl_2,
HSC_MPP,CLEC9A
HSC_MPP,CD34
CMP,CD34
ELP,CD34
HSC_MPP,SPINK2
...,...
CYCLING B/B1 CELL,CD5
CYCLING B/B1 CELL,SPN
PLASMA B CELL,JCHAIN


In [41]:
markerstest = markers.to_dict('list')

#index_names = 'anno_lvl_2', column_names = 'gene'
#'split'

In [42]:
markerstest


{'gene': ['CLEC9A',
  'CD34',
  'CD34',
  'CD34',
  'SPINK2',
  'SPINK2',
  'SPINK2',
  'MPO',
  'CSF1R',
  'IL7R',
  'KIT',
  'KIT',
  'KIT',
  'FLT3',
  'FLT3',
  'FLT3',
  'FLT3',
  'CD19',
  'CD19',
  'CD19',
  'CD19',
  'CD19',
  'CD19',
  'CD19',
  'CD19',
  'CD19',
  'CD19',
  'VPREB1',
  'VPREB1',
  'VPREB1',
  'VPREB1',
  'VPREB1',
  'VPREB1',
  'MME',
  'MME',
  'MME',
  'MME',
  'MME',
  'CDC45',
  'CDC45',
  'CDC45',
  'DHFR',
  'DHFR',
  'DHFR',
  'MKI67',
  'MKI67',
  'MKI67',
  'MKI67',
  'CD27',
  'CD27',
  'CD27',
  'RAG1',
  'RAG1',
  'RAG1',
  'RAG1',
  'RAG1',
  'DNTT',
  'DNTT',
  'MS4A1',
  'MS4A1',
  'MS4A1',
  'MS4A1',
  'MS4A1',
  'MS4A1',
  'CD24',
  'CD24',
  'CD24',
  'CD24',
  'CD24',
  'CD24',
  'CD24',
  'CD24',
  'TNFRSF17',
  'TNFRSF17',
  'TNFRSF17',
  'MME',
  'MME',
  'MME',
  'MME',
  'MME',
  'IDH2',
  'IDH2',
  'SPIB',
  'CD40',
  'CD40',
  'CD40',
  'CD40',
  'FCER2',
  'FCER2',
  'FCER2',
  'FCER2',
  'CD5',
  'SPN',
  'JCHAIN',
  'SDC1',
  'CD3

In [28]:
dict( markersoup.iloc[:, -1] )
# [: , -1] means: return all rows, return last column)


{'HSC_MPP': anno_lvl_2
 HSC_MPP    CLEC9A
 HSC_MPP      CD34
 HSC_MPP    SPINK2
 HSC_MPP       KIT
 HSC_MPP      FLT3
 Name: gene, dtype: object,
 'CMP': anno_lvl_2
 CMP      CD34
 CMP    SPINK2
 CMP       MPO
 CMP     CSF1R
 CMP       KIT
 CMP      FLT3
 Name: gene, dtype: object,
 'ELP': anno_lvl_2
 ELP      CD34
 ELP    SPINK2
 ELP      IL7R
 ELP       KIT
 ELP      FLT3
 Name: gene, dtype: object,
 'PRE PRO B CELL': anno_lvl_2
 PRE PRO B CELL      FLT3
 PRE PRO B CELL      CD19
 PRE PRO B CELL    VPREB1
 PRE PRO B CELL      RAG1
 PRE PRO B CELL     MS4A1
 Name: gene, dtype: object,
 'PRO B CELL': anno_lvl_2
 PRO B CELL      CD19
 PRO B CELL    VPREB1
 PRO B CELL       MME
 PRO B CELL     CDC45
 PRO B CELL      DHFR
 PRO B CELL     MKI67
 PRO B CELL      RAG1
 PRO B CELL      DNTT
 PRO B CELL      CD24
 PRO B CELL       MME
 Name: gene, dtype: object,
 'LATE PRO B CELL': anno_lvl_2
 LATE PRO B CELL      CD19
 LATE PRO B CELL    VPREB1
 LATE PRO B CELL       MME
 LATE PRO B CELL     

# SIMONE CODE 

In [21]:
paper_markers_list = markers["gene"].unique().tolist()
paper_markers_list

['CLEC9A',
 'CD34',
 'SPINK2',
 'MPO',
 'CSF1R',
 'IL7R',
 'KIT',
 'FLT3',
 'CD19',
 'VPREB1',
 'MME',
 'CDC45',
 'DHFR',
 'MKI67',
 'CD27',
 'RAG1',
 'DNTT',
 'MS4A1',
 'CD24',
 'TNFRSF17',
 'IDH2',
 'SPIB',
 'CD40',
 'FCER2',
 'CD5',
 'SPN',
 'JCHAIN',
 'SDC1',
 'CD38']

In [ ]:
for x in paper_markers_list:
    sc.pl.umap(pan_immune_bcells_harm, color=[x]
#               ,save = '_' + x + '_ge_c1.pdf'
              )

# ge stands for gene expression and c1 for the notebook it has been saved from.

In [ ]:
pan_immune_bcells_harm

In [ ]:
sc.pl.umap(pan_immune_bcells_harm, color="leiden", legend_loc="on data", legend_fontsize = 10
           ,save = '_leiden_c1'
          )

In [ ]:
HSC_MPP = ['CLEC9A','CD34','SPINK2','KIT','FLT3']
CMP = ['CD34','SPINK2','MPO','CSF1R','KIT','FLT3']
ELP = ['CD34', 'SPINK2','IL7R','KIT','FLT3']
PRE_PRO_B_CELL = ['FLT3','CD19','VPREB1','RAG1','MS4A1']
16	FLT3	PRE PRO B CELL
17	CD19	PRE PRO B CELL
27	VPREB1	PRE PRO B CELL
51	RAG1	PRE PRO B CELL
58	MS4A1	PRE PRO B CELL

In [ ]:
sc.pl.umap(pan_immune_bcells_harm, color=HSC_MPP, legend_loc="on data", vmax = 2
           ,save = '_HSC_MPP_c1'
          )

In [ ]:
sc.pl.dotplot(pan_immune_bcells_harm, var_names=paper_markers_list, groupby="leiden", swap_axes=True)

In [ ]:
sc.tl.draw_graph(pan_immune_bcells_harm, layout='fa')
# takes a long time (41 minutes)

In [ ]:
pan_immune_bcells_harm

In [ ]:
sc.pl.draw_graph(pan_immune_bcells_harm, layout='fa', color="leiden")

In [ ]:
pan_immune_bcells_harm

In [ ]:
# DEG output is top 500 DEGs ranked by zscore (score underlying the computation of a p-value for each gene for each group) 
sc.tl.rank_genes_groups(data, groupby=leiden, method='wilcoxon', corr_method='benjamini-hochberg', groups='all', reference='rest', 
                        n_genes=500, use_raw=False, log_transformed=True)

# Reformat output so easier to understand. The resultant df shows the log2 fold change of the ln-transformed data for each gene ordered by z-score 
result = data.uns['rank_genes_groups']
groups = result['names'].dtype.names
DE = pd.DataFrame(
    {group + '_' + key: result[key][group]
    for group in groups for key in ['names', 'pvals','pvals_adj','logfoldchanges']}).head(n_genes_to_calc)

# Save output (Default scanpy output as pandas) 
DE.to_csv("/path/to/filename.csv")

In [ ]:
'''Error leiden column is present however the data isn't logarithmized? according to the error. Look back to old files and uncover the logarithmize function used. '''

In [ ]:
help(reorder_categories)

In [ ]:
"""
adata.obs["annots_SW_lvl2"] = adata.obs["annots_SW_lvl2"].cat.reorder_categories(new_order_list)

To organise your columns so that they are properly ordered in the outputs. e.g. putting in the leiden column then




"""

# SIMONE CODE NO MORE

In [ ]:
pan_immune_bcells_harm.uns['log1p']["base"] = None
#this works for some mysterious reason 

In [ ]:
pan_immune_bcells_harm

In [ ]:
pan_immune_bcells_harm.X

In [ ]:
sc.pp.scale(pan_immune_bcells_harm,max_value=10)

In [ ]:
# DEG output is top 500 DEGs ranked by zscore (score underlying the computation of a p-value for each gene for each group) 
sc.tl.rank_genes_groups(pan_immune_bcells_harm, groupby='leiden', method='wilcoxon', corr_method='benjamini-hochberg', 
                        groups='all', reference='rest', n_genes=500, use_raw=False, log_transformed=True, zero_center = False)

# Reformat output so easier to understand. The resultant df shows the log2 fold change of the ln-transformed data for each gene ordered by z-score 
result = pan_immune_bcells_harm.uns['rank_genes_groups']
groups = result['names'].dtype.names
DE = pd.DataFrame(
    {group + '_' + key: result[key][group]
    for group in groups for key in ['names', 'pvals','pvals_adj','logfoldchanges']}).head(50)

# Save output (Default scanpy output as pandas) 
DE.to_csv(writes + "DE.csv")
#"C:\Users/sambe/OneDrive - Newcastle University/Research projects/python/write
display(DE)

In [ ]:
''' 
https://github.com/scverse/scanpy/issues/653 
use .copy for subsetting the B cell datasets
sc.pp.scale <- potentially for scaling the data to 10 so that there is no value  
DO NOT CUT DATA THAT IS NEGATIVE or maybe do but to lose infomation is not generally a good idea.
The error is probably from the matrix in which a negative value of expression is divided by a postitive value to find the expression difference,
this forms an error 

This comment is in place as majority of data was returned to blank due to to negative values causing a gene ranking error 

'''

In [ ]:
'''
I am confused about how to progress with the marker genes and how they apply

In [ ]:
DE

In [ ]:
rank_genes_groups

In [ ]:
#string concatenation
name = "JIM"
age = "CAT"
"hello i'm %s, i'm %s years old" % (name, age)